<a href="https://colab.research.google.com/github/mirdbg/FinTok/blob/main/notebooks/02_audio/02_Audio_Final.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# FinTok · Audio pipeline final

Genera **un WAV por escena** usando F5-TTS.

La decisión de quién habla ya viene del JSON producido por `src/text`.  
`src/audio` **no asigna voces por número de escena**.

Ejemplo:

```json
"speaker": "ANDREA"
```

selecciona la referencia de Andrea, mientras que:

```json
"speaker": "MIRIAM"
```

selecciona la referencia de Miriam.

El valor original de `speaker` se conserva en el JSON de salida.


## 1. Clonar o actualizar FinTok

Pon la URL real de tu repositorio en `REPO_URL`.

In [13]:
from pathlib import Path

repo_path = Path("/content/FinTok")

if repo_path.exists():
    print("✓ FinTok ya está clonado. Sincronizando con GitHub...")

    %cd /content/FinTok

    !git fetch origin
    !git reset --hard origin/main
    !git clean -fd

    %cd /content

    print("✓ FinTok actualizado desde GitHub")

else:
    print("Clonando FinTok...")
    !git clone https://github.com/mirdbg/FinTok.git /content/FinTok


✓ FinTok ya está clonado. Sincronizando con GitHub...
/content/FinTok
remote: Enumerating objects: 9, done.
remote: Counting objects: 100% (9/9), done.
remote: Compressing objects: 100% (5/5), done.
remote: Total 5 (delta 2), reused 0 (delta 0), pack-reused 0 (from 0)
Unpacking objects: 100% (5/5), 1.07 KiB | 1.07 MiB/s, done.
From https://github.com/mirdbg/FinTok
   299f9f9..1cb8a2e  main       -> origin/main
HEAD is now at 1cb8a2e Update example_02.json
/content
✓ FinTok actualizado desde GitHub


## 2. Instalar dependencias

In [9]:
import os

# Evita el error de PYTHONHASHSEED observado en Colab.
os.environ["PYTHONHASHSEED"] = "0"

!apt-get -qq update
!apt-get -qq install -y ffmpeg
!pip -q install -U f5-tts soundfile

print("✓ Dependencias instaladas")


W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu noble InRelease' does not seem to provide it (sources.list entry misspelt?)
✓ Dependencias instaladas


## 3. Comprobar referencias de voz

El pipeline espera estos WAV:

- Andrea: `data/audio_samples/03_EN_ANDREA_SHORT_FAST.wav`
- Miriam: `data/audio_samples/06_EN_MIRIAM_SHORT_FAST.wav`


In [10]:
from pathlib import Path
from IPython.display import Audio, display

REPO_DIR = Path("/content/FinTok")

references = {
    "ANDREA": REPO_DIR / "data/audio_samples/03_EN_ANDREA_SHORT_FAST.wav",
    "MIRIAM": REPO_DIR / "data/audio_samples/06_EN_MIRIAM_SHORT_FAST.wav",
}

for speaker, path in references.items():
    if not path.exists():
        raise FileNotFoundError(f"No encuentro {speaker}: {path}")

    print(f"✓ {speaker}: {path}")
    display(Audio(filename=str(path)))


✓ ANDREA: /content/FinTok/data/audio_samples/03_EN_ANDREA_SHORT_FAST.wav


✓ MIRIAM: /content/FinTok/data/audio_samples/06_EN_MIRIAM_SHORT_FAST.wav


## 4. Seleccionar el JSON generado por texto

El JSON debe contener `speaker` en cada escena.

In [14]:
INPUT_JSON = "/content/FinTok/outputs/text/example_02.json"

print("JSON de entrada:", INPUT_JSON)


JSON de entrada: /content/FinTok/outputs/text/example_02.json


## 5. Revisar qué voz se usará

Esta comprobación se hace **antes de cargar F5-TTS**.

In [15]:
import json

with open(INPUT_JSON, "r", encoding="utf-8") as f:
    preview = json.load(f)

for scene in preview["scenes"]:
    print(
        f"scene {scene['scene_id']} | "
        f"{scene['scene_type']} | "
        f"speaker={scene['speaker']}"
    )


scene 1 | hook | speaker=ANDREA
scene 2 | explanation | speaker=MIRIAM
scene 3 | explanation | speaker=MIRIAM
scene 4 | outro | speaker=ANDREA


## 6. Generar los audios

El notebook llama al código real de `src/audio/audio_generator.py`.

In [16]:
import sys

sys.path.insert(0, "/content/FinTok")

from src.audio.audio_generator import generate_audio_from_json

result = generate_audio_from_json(
    input_json=INPUT_JSON,
    speed=1.0,
)

print("\n✓ Pipeline de audio terminado")


Loading F5-TTS...
Download Vocos from huggingface charactr/vocos-mel-24khz

vocab :  /usr/local/lib/python3.13/dist-packages/f5_tts/infer/examples/vocab.txt
token :  custom
model :  /root/.cache/huggingface/hub/models--SWivid--F5-TTS/snapshots/84e5a410d9cead4de2f847e7c9369a6440bdfaca/F5TTS_v1_Base/model_1250000.safetensors 

[1] hook | speaker=ANDREA | Did you know that Apple spent a massive $89.3 billion in 2025 just buying back its own sto
Converting audio...
Using custom reference text...

ref_text   Welcome back. Stock prices are moving and investors are asking the same question. What happens next? Let's get started. 
gen_text 0 Did you know that Apple spent a massive $89.3 billion in 2025 just buying back its own stock and paying dividends?


Generating audio in 1 batches...


100%|██████████| 1/1 [00:04<00:00,  4.54s/it]


[2] explanation | speaker=MIRIAM | This is FinTok — your daily dose of financial knowledge.
Converting audio...
No reference text provided, transcribing reference audio...


[transformers] `torch_dtype` is deprecated! Use `dtype` instead!


Loading weights:   0%|          | 0/587 [00:00<?, ?it/s]

...

[3] explanation | speaker=MIRIAM | During 2025, Apple repurchased $89.3 billion of its common stock and paid dividends and di
Converting audio...
Using cached preprocessed reference audio...
Using cached reference text...

ref_text   Welcome back! Stock prices are moving and investors are asking the same question. What happens next? Let's get started!. 
gen_text 0 During 2025, Apple repurchased $89.3 billion of its common stock and paid dividends and dividend equivalents of $15.4 billion, as detailed in their 2025 filing.
gen_text 1 This significant amount demonstrates the company’s financial health and its commitment to returning value to shareholders. They also raised their quarterly dividend from $0.25 to $0.26 per share.


Generating audio in 2 batches...


100%|██████████| 2/2 [00:09<00:00,  4.53s/it]


[4] outro | speaker=ANDREA | That's your FinTok for today. See you in the next one.
Converting audio...
Using cached preprocessed reference audio...
Using custom reference text...

ref_text   Welcome back. Stock prices are moving and investors are asking the same question. What happens next? Let's get started. 
gen_text 0 That's your FinTok for today. See you in the next one.


Generating audio in 1 batches...


100%|██████████| 1/1 [00:02<00:00,  2.59s/it]


✓ Generated 4 audio files
✓ Audio directory: /content/FinTok/data/audio_output/AAPL_2025
✓ Updated JSON: /content/FinTok/data/audio_output/AAPL_2025/script_with_audio.json

✓ Pipeline de audio terminado


## 7. Escuchar las cuatro escenas

In [17]:
from pathlib import Path
from IPython.display import Audio, display

repo_root = Path("/content/FinTok")

for scene in result["scenes"]:
    print(
        f"Escena {scene['scene_id']} | "
        f"{scene['scene_type']} | "
        f"{scene['speaker']}"
    )
    print(scene["text"])
    print("Audio:", scene["audio_path"])

    audio_path = Path(scene["audio_path"])
    if not audio_path.is_absolute():
        audio_path = repo_root / audio_path

    display(Audio(filename=str(audio_path)))
    print("-" * 80)


Escena 1 | hook | ANDREA
Did you know that Apple spent a massive $89.3 billion in 2025 just buying back its own stock and paying dividends?
Audio: data/audio_output/AAPL_2025/scene_01_hook_andrea.wav


--------------------------------------------------------------------------------
Escena 2 | explanation | MIRIAM
This is FinTok — your daily dose of financial knowledge.
Audio: data/audio_output/AAPL_2025/scene_02_explanation_miriam.wav


--------------------------------------------------------------------------------
Escena 3 | explanation | MIRIAM
During 2025, Apple repurchased $89.3 billion of its common stock and paid dividends and dividend equivalents of $15.4 billion, as detailed in their 2025 filing. This significant amount demonstrates the company’s financial health and its commitment to returning value to shareholders. They also raised their quarterly dividend from $0.25 to $0.26 per share.
Audio: data/audio_output/AAPL_2025/scene_03_explanation_miriam.wav


--------------------------------------------------------------------------------
Escena 4 | outro | ANDREA
That's your FinTok for today. See you in the next one.
Audio: data/audio_output/AAPL_2025/scene_04_outro_andrea.wav


--------------------------------------------------------------------------------


In [ ]:
from pathlib import Path
import shutil

repo_root = Path("/content/FinTok")

# Carpeta de salida dentro del repositorio
output_dir = repo_root / "outputs" / "audio" / "example_01"
output_dir.mkdir(parents=True, exist_ok=True)

for scene in result["scenes"]:
    scene_id = int(scene["scene_id"])

    # Ruta del audio generado por el pipeline
    source_path = Path(scene["audio_path"])

    if not source_path.is_absolute():
        source_path = repo_root / source_path

    # Nombre final
    destination_path = output_dir / f"audio_{scene_id:02d}.wav"

    shutil.copy2(source_path, destination_path)

    print(f"✓ {destination_path}")

print("\n✓ Audios guardados en:")
print(output_dir)

## 8. Ejecución fuera de Colab

Desde la raíz de FinTok:

```bash
python -m src.audio.audio_generator data/outputs/script.json
```

Opcionalmente:

```bash
python -m src.audio.audio_generator data/outputs/script.json \
    --output-dir data/audio_output/AAPL_2024 \
    --output-json data/audio_output/AAPL_2024/script_with_audio.json \
    --speed 1.0
```

El flujo queda:

`src/text → JSON con speaker → src/audio → 1 WAV por scene → JSON con audio_path`
